# Whale markets, LTV distribution, and zoomed LTV history

GitLab **#68** (Krzys WhatsApp follow-up):

1. From which markets are these whales?
2. LTV distribution per whale (min / median / max, etc.)
3. Historical LTV charts for a representative subset, with a **zoomed** y-axis (not 0–100%)

Outputs land in `output/`.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

PANEL = Path("../../feature_engineering/data/daily_features_all.csv")
OUT = Path("output")
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PANEL, parse_dates=["date"])
print(df.shape)
print(sorted(df["whale_id"].unique(), key=lambda x: int(x.split("-")[1])))

## Markets and LTV distribution

One row per whale. LTV stats use **open-debt days** only (`ltv` not null and `borrow_balance_usd >= 1`).

In [ ]:
active = df[df["ltv"].notna() & (df["borrow_balance_usd"].fillna(0) >= 1)].copy()

rows = []
for whale_id, g in df.groupby("whale_id", sort=False):
    proto = g["protocol"].mode().iloc[0]
    coll = g["collateral_token"].mode().iloc[0]
    borr = g["borrow_token"].mode().iloc[0]
    a = active.loc[active["whale_id"] == whale_id, "ltv"]
    if len(a) == 0:
        stats = dict(
            ltv_min=pd.NA, ltv_p25=pd.NA, ltv_median=pd.NA, ltv_p75=pd.NA,
            ltv_max=pd.NA, ltv_mean=pd.NA, n_days_open_debt=0,
        )
    else:
        stats = dict(
            ltv_min=round(a.min(), 4),
            ltv_p25=round(a.quantile(0.25), 4),
            ltv_median=round(a.median(), 4),
            ltv_p75=round(a.quantile(0.75), 4),
            ltv_max=round(a.max(), 4),
            ltv_mean=round(a.mean(), 4),
            n_days_open_debt=int(len(a)),
        )
    rows.append({
        "whale_id": whale_id,
        "protocol": proto,
        "market": f"{coll} / {borr}",
        "collateral_token": coll,
        "borrow_token": borr,
        **stats,
        "n_days_panel": int(len(g)),
    })

summary = pd.DataFrame(rows)
summary["_n"] = summary["whale_id"].str.split("-").str[1].astype(int)
summary = summary.sort_values("_n").drop(columns="_n")
summary_path = OUT / "whale_markets_ltv_summary.csv"
summary.to_csv(summary_path, index=False)
print("wrote", summary_path)
summary

## Zoomed historical LTV charts

Representative set: **W-12**, **W-1** (high/tight), **W-3** (active/mixed), **W-11** (low/variable), **W-17** (Aave), **W-5** (syrup).

Y-axis is zoomed to each whale’s open-debt LTV band (plus LLTV reference). No in-plot title — identity is in the filename / footer.

In [ ]:
WHALES = ["W-12", "W-1", "W-3", "W-11", "W-17", "W-5"]

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#333333",
    "axes.grid": True,
    "grid.color": "#dddddd",
    "grid.linewidth": 0.6,
    "font.size": 11,
})

for wid in WHALES:
    g = df[df["whale_id"] == wid].sort_values("date")
    plot = g[g["ltv"].notna() & (g["borrow_balance_usd"].fillna(0) >= 1)].copy()
    if plot.empty:
        print("skip empty", wid)
        continue

    ltv_min = float(plot["ltv"].min())
    ltv_max = float(plot["ltv"].max())
    pad = max(0.005, (ltv_max - ltv_min) * 0.15)
    ymin = max(0.0, ltv_min - pad)
    ymax = min(1.0, ltv_max + pad)
    if ymax - ymin < 0.02:
        mid = (ymin + ymax) / 2
        ymin = max(0.0, mid - 0.01)
        ymax = min(1.0, mid + 0.01)

    lltv = None
    if plot["liquidation_threshold"].notna().any():
        lltv = float(plot["liquidation_threshold"].dropna().iloc[0])
        ymax = max(ymax, min(1.0, lltv + 0.005))

    fig, ax = plt.subplots(figsize=(9, 3.8))
    ax.plot(plot["date"], plot["ltv"], color="#4C72B0", linewidth=1.4)
    if lltv is not None:
        ax.axhline(lltv, color="#C44E52", linestyle="--", linewidth=1.0, label=f"LLTV {lltv:.3f}")
        ax.legend(frameon=False, loc="best")

    ax.set_ylim(ymin, ymax)
    ax.set_ylabel("LTV")
    ax.set_xlabel("Date")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    fig.autofmt_xdate(rotation=0, ha="center")

    market = f"{plot['collateral_token'].iloc[0]} / {plot['borrow_token'].iloc[0]}"
    proto = plot["protocol"].iloc[0]
    fig.text(
        0.01, 0.02,
        f"{wid} · {proto} · {market} (open-debt days)",
        fontsize=9, color="#555555", transform=fig.transFigure,
    )
    fig.tight_layout(rect=[0, 0.06, 1, 1])
    path = OUT / f"ltv_history_{wid}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)
    print("wrote", path, f"ylim=({ymin:.3f},{ymax:.3f}) n={len(plot)}")